# Original coursework: sanitized publication copy

This is a sanitized copy of the recovered original course notebook, not a newly reconstructed demo. Outputs, execution counts and incidental metadata were cleared before publication to avoid exposing customer review excerpts, local paths and account information. The source has not been rerun here against its original dataset. Team credits and original AI-use disclosure remain. Any credentials must be supplied privately. The original notebook includes an unfinished LLM section; its presence is not proof of a completed evaluation. Name-inferred gender labels are noisy proxies, not observed demographic truth.


# 1. Team Information

**Team ID:** Automotive Repair
**Team Members:** Christa Irakoze, Saurabh Bhatkar, Basilio Johnson

*This notebook contains all required sections: Project Overview, Data Sources, Data Description and Summary Statistics, Data Quality and Suitability, Preliminary Data Exploration, Proposed AI Solution, and Generative AI Use.*

---

# 2. Project Overview

This project examines whether Yelp reviews of automotive repair businesses show a measurable gender gap — between inferred-female and inferred-male reviewers — in how often they mention upselling or sales pressure, and in the sentiment of those mentions. Automotive repair is a market defined by information asymmetry: most customers cannot independently verify whether a recommended repair is genuinely necessary, which creates an opening for upselling. Anecdotal and journalistic claims suggest women are disproportionately targeted for unnecessary upsells, but this is rarely tested systematically at scale. Using text mining and sentiment analysis on real Yelp review text, combined with a name-based gender inference on reviewers, the goal is not to prove discriminatory intent, but to surface measurable, data-driven patterns that repair shop operators, franchise quality-assurance teams, and consumer-advocacy organizations could use to identify practices or locations meriting further review.

# 3. Data Sources

**Source:** [Yelp Open Dataset](https://www.yelp.com/dataset), downloaded directly as a zip archive from yelp.com.

**Files used:**
- `yelp_academic_dataset_business.json` — 150,346 businesses total. Structured fields (name, address, city, state, star rating, review count, category tags, attributes, hours).
- `yelp_academic_dataset_review.json` — 6,990,280 reviews total. Unstructured review text plus star rating, useful/funny/cool vote counts, and date.
- `yelp_academic_dataset_user.json` — 1,987,891 users total. Used only for the reviewer's first `name` (no gender field exists in the dataset — gender must be inferred).

The script below reproduces exactly how `business_800_sample.csv` and `reviews_17515_sample_with_gender.csv` (the two working files used for the rest of this notebook) were built from the three raw files above, following the same `pandas`/`pd.read_json` approach demonstrated in the course's `yelp_exploration.ipynb`.

In [ ]:
%pip install -q pandas matplotlib nltk gender-guesser scikit-learn scipy vaderSentiment requests

In [ ]:
# Load the raw business file directly -- it's small enough (118MB) to fit
# in memory all at once, the same way yelp_exploration.ipynb loads its data.
import pandas as pd

business_path = "yelp_academic_dataset_business.json"
review_path = "yelp_academic_dataset_review.json"
user_path = "yelp_academic_dataset_user.json"

# business.json (118MB, 150,346 rows) is small enough to load directly with
# pd.read_json, the same way yelp_exploration.ipynb loads the course dataset.
business_df_full = pd.read_json(business_path, lines=True)
print(f"Total businesses: {len(business_df_full):,}")
business_df_full.head()

`review.json` (~5.3GB) and `user.json` (~3.36GB) are far larger than the course's sample dataset and cannot be loaded in full with `pd.read_json` the way `yelp_exploration.ipynb` does — that would require holding several gigabytes in memory at once just to keep a small fraction of the rows. Instead, both files are streamed line-by-line below: each line is checked against the set of business/user IDs we actually need *before* being parsed as JSON, so memory use stays small regardless of file size. This runs in well under a minute despite the file sizes.

**How the data were filtered:**

1. Filter `business.json` to businesses whose `categories` contain **"Auto Repair"**.
2. Restrict to **Pennsylvania (PA)** — the largest single-metro pool available (1,245 businesses, more than any other state/metro in the dataset) — to keep the sample geographically coherent rather than scattered nationally.
3. Randomly sample **800** of those 1,245 businesses (fixed seed = 42, for reproducibility).
4. Stream `review.json` once, keeping only reviews whose `business_id` is in the 800-business set.
5. Stream `user.json` once, keeping only the first `name` for each unique reviewer behind those reviews.
6. Apply the gender-inference method (built here, justified in Section 5.1) to each reviewer's first name, and drop any review whose name could not be confidently resolved.

In [ ]:
# Step 1: keep only Auto Repair businesses, then only the ones in PA.
import random

auto_mask = business_df_full["categories"].fillna("").str.contains("Auto Repair")
auto_df = business_df_full[auto_mask].copy()
print(f"Auto Repair businesses (all areas): {len(auto_df):,}")

pa_auto_df = auto_df[auto_df["state"] == "PA"].copy()
print(f"Auto Repair businesses in PA: {len(pa_auto_df):,}")

# Randomly pick 800 of them (seed=42 so anyone re-running this gets the same 800).
random.seed(42)
business_df = pa_auto_df.sample(n=800, random_state=42).copy()
sample_ids = set(business_df["business_id"])
print(f"Sampled {len(sample_ids)} PA business ids (seed=42)")

business_df.to_csv("business_800_sample.csv", index=False)
print("Wrote business_800_sample.csv")

In [ ]:
# Step 2 (part A): build the gender-inference method used later on reviewer
# names. Two pieces: a curated name lookup (gender-guesser) and a backup
# classifier (NLTK Naive Bayes) for names the lookup doesn't recognize.
import nltk
from nltk.corpus import names
from nltk.classify import NaiveBayesClassifier, accuracy
import gender_guesser.detector as gg

nltk.download("names", quiet=True)


def name_features(name):
    # Turns a name into the letter patterns the classifier learns from
    # (e.g. names ending in "a" skew female, endings in "-o" skew male).
    name = name.lower()
    return {
        "last_letter": name[-1],
        "last_2": name[-2:],
        "last_3": name[-3:],
        "first_letter": name[0],
        "first_2": name[:2],
    }


# Load NLTK's built-in list of ~8,000 names already labeled male/female.
male_names = [(n, "male") for n in names.words("male.txt")]
female_names = [(n, "female") for n in names.words("female.txt")]
labeled_names = male_names + female_names

# Hold out 10% of names to honestly test the classifier on names it never
# trained on, instead of just checking it against its own training data.
random.seed(42)
random.shuffle(labeled_names)
featuresets = [(name_features(n), g) for (n, g) in labeled_names]
cut = int(len(featuresets) * 0.9)
train_set, test_set = featuresets[:cut], featuresets[cut:]

nb_classifier = NaiveBayesClassifier.train(train_set)
nb_self_accuracy = accuracy(nb_classifier, test_set)
print(f"NLTK Naive Bayes held-out accuracy (same-corpus test split): {nb_self_accuracy:.3f}")

detector = gg.Detector(case_sensitive=False)


def gg_predict(name):
    # Looks the name up in gender-guesser's curated table.
    # Returns one of: male, female, mostly_male, mostly_female, andy, unknown.
    return detector.get_gender(name)


def nb_predict(name, threshold=0.90):
    # Asks the NLTK classifier for a male/female probability, but only
    # accepts the answer if it's confident enough (>= threshold).
    dist = nb_classifier.prob_classify(name_features(name))
    male_p, female_p = dist.prob("male"), dist.prob("female")
    top_label = "male" if male_p > female_p else "female"
    top_p = max(male_p, female_p)
    if top_p < threshold:
        return "unisex/ambiguous", round(top_p, 2)
    return top_label, round(top_p, 2)


def infer_gender(name, nb_fallback_threshold=0.90):
    """
    Primary: gender-guesser (curated lookup).
    Fallback: NLTK character n-gram Naive Bayes, only when gender-guesser
    has no answer, and only above a strict confidence threshold.
    Returns 'male', 'female', or 'unresolved'. Justified in Section 5.1.
    """
    if not name or not str(name).strip():
        return "unresolved"
    first_name = str(name).split()[0]
    gg_result = gg_predict(first_name)
    if gg_result in ("male", "mostly_male"):
        return "male"
    if gg_result in ("female", "mostly_female"):
        return "female"
    # gender-guesser had no answer -- try the stricter backup classifier.
    nb_label, nb_p = nb_predict(first_name, threshold=nb_fallback_threshold)
    if nb_label in ("male", "female"):
        return nb_label
    return "unresolved"

**Note on the accuracy number above:** it is measured on a held-out split of the *same* NLTK names corpus the model trained on. It tells us the model can distinguish patterns within that corpus, but not how well it generalizes to messy, real-world Yelp first names — Section 5.1 tests that directly and is why the classifier is only used as a fallback, not the primary method.

In [ ]:
# Step 2 (part B): stream review.json one line at a time, keeping only
# reviews for our 800 sampled businesses. Reading line-by-line (instead of
# loading the whole 5.3GB file) keeps memory use small.
import json
import re
import time

business_id_re = re.compile(r'"business_id":"([^"]+)"')
matched_reviews = []
line_count = 0
t0 = time.time()
with open(review_path, "r", encoding="utf-8") as f:
    for line in f:
        line_count += 1
        # Check the business_id before doing a full (slower) JSON parse.
        m = business_id_re.search(line)
        if m and m.group(1) in sample_ids:
            try:
                matched_reviews.append(json.loads(line))
            except json.JSONDecodeError:
                pass

print(f"Scanned {line_count:,} review lines in {time.time() - t0:.0f}s")
print(f"Matched reviews: {len(matched_reviews):,}")

reviews_df = pd.DataFrame(matched_reviews)

In [ ]:
# Step 3: stream user.json the same way, but this time we only need the
# first `name` field for each reviewer who showed up in reviews_df above.
needed_user_ids = set(reviews_df["user_id"].unique())
print(f"Unique reviewers to look up: {len(needed_user_ids):,}")

user_id_re = re.compile(r'"user_id":"([^"]+)"')
name_re = re.compile(r'"name":"((?:[^"\\]|\\.)*)"')
matched_users = {}
line_count = 0
t0 = time.time()
with open(user_path, "r", encoding="utf-8") as f:
    for line in f:
        line_count += 1
        m = user_id_re.search(line)
        if m and m.group(1) in needed_user_ids:
            uid = m.group(1)
            nm = name_re.search(line)
            matched_users[uid] = nm.group(1) if nm else None
        # Stop early once every reviewer we need has been found.
        if len(matched_users) == len(needed_user_ids):
            break

print(f"Scanned {line_count:,} user lines in {time.time() - t0:.0f}s")
print(f"Matched users: {len(matched_users):,} / {len(needed_user_ids):,}")

reviews_df["reviewer_name"] = reviews_df["user_id"].map(matched_users)

In [ ]:
# Step 4: guess each reviewer's gender from their first name, then drop
# any review where that guess couldn't be made confidently.
reviews_df["inferred_gender"] = reviews_df["reviewer_name"].apply(infer_gender)

n_total = len(reviews_df)
resolved_df = reviews_df[reviews_df["inferred_gender"] != "unresolved"].copy()
n_dropped = n_total - len(resolved_df)

print(f"Matched reviews before gender filtering: {n_total:,}")
print(f"Dropped {n_dropped:,} as unresolved ({n_dropped / n_total:.1%})")
print(resolved_df["inferred_gender"].value_counts())

# Named after the 17,515 matched reviews this script starts with, before the
# gender-unresolved rows are dropped (final row count is smaller, at 16,735).
resolved_df.to_csv("reviews_17515_sample_with_gender.csv", index=False)
print("Wrote reviews_17515_sample_with_gender.csv")

# 4. Data Description and Summary Statistics

In [ ]:
# Reload the two working files fresh from disk, so this section can be
# re-run on its own without re-running the (slower) Section 3 pipeline.
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

business_df = pd.read_csv("business_800_sample.csv")
reviews_df = pd.read_csv("reviews_17515_sample_with_gender.csv")

print("Sampled businesses:", business_df.shape)
print("Gender-resolved reviews:", reviews_df.shape)

### 4.1 Variables and missing values

In [ ]:
# Column types and missing-value counts for the business table.
print("BUSINESS TABLE")
print(business_df.dtypes)
print("\nMissing values:")
print(business_df.isnull().sum())

In [ ]:
# Column types and missing-value counts for the review table.
print("REVIEW TABLE")
print(reviews_df.dtypes)
print("\nMissing values:")
print(reviews_df.isnull().sum())

### 4.2 Star rating distribution

In [ ]:
# How review star ratings are distributed across the sample.
reviews_df["stars"].value_counts().sort_index().plot(kind="bar", color="#4C72B0")
plt.title("Review Star Rating Distribution (800 Auto Repair Businesses)")
plt.xlabel("Stars")
plt.ylabel("Number of Reviews")
plt.show()

print(reviews_df["stars"].describe())

### 4.3 Review length (text data statistics)

In [ ]:
# Word count per review, to see how much text each review typically has.
reviews_df["word_count"] = reviews_df["text"].astype(str).str.split().str.len()

reviews_df["word_count"].plot(kind="hist", bins=50, color="#55A868")
plt.title("Review Length Distribution (words per review)")
plt.xlabel("Word count")
plt.ylabel("Number of Reviews")
plt.show()

print(reviews_df["word_count"].describe())

In [ ]:
# Build a bag-of-words vocabulary (stopwords removed) to see how many
# distinct words actually appear across all the reviews.
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer

token = RegexpTokenizer(r"[a-zA-Z]+")
cv = CountVectorizer(stop_words="english", tokenizer=token.tokenize, lowercase=True)
cv.fit(reviews_df["text"].astype(str))

print(f"Number of documents (reviews): {len(reviews_df):,}")
print(f"Vocabulary size (unique tokens, stopwords removed): {len(cv.vocabulary_):,}")

### 4.4 Businesses by city (within PA)

In [ ]:
# Confirm every sampled business really is in PA, then see which cities
# within PA they're concentrated in.
city_counts = business_df["city"].value_counts().head(15)
city_counts.plot(kind="bar", color="#C44E52")
plt.title("Sampled PA Auto Repair Businesses by City (top 15)")
plt.xlabel("City")
plt.ylabel("Number of Businesses")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("All sampled businesses are in state:", business_df["state"].unique())
print()
print(city_counts)

### 4.5 Reviews per business

In [ ]:
# How many reviews each business has -- checks whether a few businesses
# dominate the sample or reviews are spread out fairly evenly.
reviews_per_business = reviews_df.groupby("business_id").size()

reviews_per_business.plot(kind="hist", bins=50, color="#8172B2")
plt.title("Reviews per Business (800-business sample)")
plt.xlabel("Number of reviews")
plt.ylabel("Number of businesses")
plt.show()

print(reviews_per_business.describe())

### 4.6 Reviews by inferred reviewer gender

In [ ]:
# Recap of the male/female split among reviews we could confidently
# assign a gender to (built in Section 3, Step 4).
gender_counts = reviews_df["inferred_gender"].value_counts()
gender_counts.plot(kind="bar", color=["#4C72B0", "#DD8452"])
plt.title("Reviews by Inferred Reviewer Gender")
plt.xlabel("Inferred Gender")
plt.ylabel("Number of Reviews")
plt.show()

print(gender_counts)
print(reviews_df["inferred_gender"].value_counts(normalize=True))

# 5. Data Quality and Suitability

This dataset is well suited to the business question, but several limitations shape what conclusions can be drawn:

- **Reviewer gender is inferred, not observed.** Yelp records no gender field. The method (built in Section 3) was chosen after comparing it against alternatives — Section 5.1 documents that comparison and its failure modes.
- **Sample size:** 800 of 1,245 PA Auto Repair businesses were randomly sampled (64.3%), yielding 17,515 reviews and, after dropping unresolved-gender reviews, 16,735 usable records — ample for the comparative analysis planned in the next stage.
- **Single-area by design:** all 800 businesses are in Pennsylvania (the Greater Philadelphia metro, in this dataset), chosen deliberately over a scattered multi-state sample so that regional differences in pricing, regulation, and urban/rural mix don't get mistaken for a gender effect. The tradeoff is that findings describe this one metro area and should not be generalized nationally without further validation elsewhere.
- **Self-selection bias:** Yelp reviewers are not a random sample of all customers; people who leave reviews (positive or negative) may differ systematically from those who don't, in ways that could correlate with gender.
- **Gender split is not even:** among gender-resolved reviews, 57% are inferred male vs. 43% female — worth investigating further rather than assuming a data error, since the underlying reviewer pool itself may skew this way for auto-repair businesses specifically.

## 5.1 Gender Inference Method Selection

The classifier and `infer_gender()` function used to build `reviews_17515_sample_with_gender.csv` in Section 3 are reused here (not retrained) to compare `gender-guesser` against the NLTK Naive Bayes fallback on its own, and justify why the hybrid — rather than either method alone — was used.

### 5.1.1 Head-to-head comparison on real reviewer first names

These are first names pulled from actual reviewers of automotive-repair businesses in our PA sample (joined from `review.json` → `user.json` on `user_id` in Section 3).

In [ ]:
# Run both methods on the same real names and compare side by side.
sample_reviewer_names = [
    "Meghan", "Gayle", "Kristin", "Daniel", "Stacy", "Gaemus", "Robert",
    "Vinnie", "Michael", "Sunni", "Kristie", "Meg", "Jessica", "Corie",
    "John", "Eric", "L", "Lola", "Anna", "Nicole", "Taylor", "Xman",
    "Emily", "Mike", "Angela",
]

import pandas as pd

rows = []
for n in sample_reviewer_names:
    gg_result = gg_predict(n)
    nb_label, nb_p = nb_predict(n, threshold=0.80)
    rows.append({"name": n, "gender_guesser": gg_result, "nltk_nb_label": nb_label, "nltk_nb_prob": nb_p})

comparison_df = pd.DataFrame(rows)
comparison_df

### 5.1.2 Interpretation

Several disagreements are informative:

- **`Daniel`, `Michael`, `Mike`, `Meg`** — unambiguous, extremely common names that `gender-guesser` resolves correctly (`male`, `male`, `male`, `female`), but the NB classifier drops as `unisex/ambiguous` because the probability margin from character endings alone is weak.
- **`Meghan` → male (0.85)`, `Vinnie` → female (0.95)`, `Sunni` → female (1.0)`** — the NB classifier is *confidently wrong* on these. A wrong label stated with high confidence is worse for this project than an honest "unknown," because it silently corrupts the gender-comparison groups instead of just shrinking the sample.
- **`Gaemus`, `Xman`** — `gender-guesser` correctly reports `unknown` (not real/common first names); the NB classifier invents a confident label for them because it can't recognize "not a name."

**Decision:** use `gender-guesser` as the primary method. Only when it returns `unknown` do we fall back to the NB classifier — and we raise its acceptance threshold to **0.90** (rather than 0.80) for the fallback specifically, since the comparison above shows 0.80 is not strict enough to avoid confidently wrong calls. This keeps the NB model useful for its actual strength (covering names absent from the `gender-guesser` table) without letting it override a curated method on names it already gets right.

### 5.1.3 Resulting coverage on the sample

In [ ]:
# infer_gender() was already defined in Section 3 -- reused here, not
# redefined. Apply it to the sample and drop anything unresolved.
results = [infer_gender(n) for n in sample_reviewer_names]
coverage_df = pd.DataFrame({"name": sample_reviewer_names, "inferred_gender": results})

n_total = len(coverage_df)
resolved_sample_df = coverage_df[coverage_df["inferred_gender"] != "unresolved"].reset_index(drop=True)
n_dropped = n_total - len(resolved_sample_df)

print(f"Dropped {n_dropped} of {n_total} names as unresolved "
      f"({n_dropped / n_total:.1%}) -- excluded from downstream gender analysis.")
print(resolved_sample_df["inferred_gender"].value_counts())
resolved_sample_df

# 6. Preliminary Data Exploration

This section applies sentiment analysis and keyword analysis to the 16,735 gender-resolved reviews (Section 3), directly targeting the business question: do upsell/pressure mentions and their sentiment differ between inferred-male and inferred-female reviewers?

## 6.1 Sentiment Analysis (VADER)

VADER (Valence Aware Dictionary and sEntiment Reasoner) is the same lexicon-based sentiment tool used in the course's `Sentiment_Analysis.ipynb`. It scores text from -1 (most negative) to +1 (most positive) via a single `compound` score, and the standard thresholds (used in that notebook too) are:
- **positive:** compound >= 0.05
- **neutral:** -0.05 < compound < 0.05
- **negative:** compound <= -0.05

In [ ]:
# Score every review's text with VADER, then bucket it into
# positive / neutral / negative using the standard thresholds.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()


def vader_compound(text):
    return analyzer.polarity_scores(str(text))["compound"]


def sentiment_label(compound):
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"


reviews_df["compound_sentiment"] = reviews_df["text"].apply(vader_compound)
reviews_df["sentiment_label"] = reviews_df["compound_sentiment"].apply(sentiment_label)

reviews_df["sentiment_label"].value_counts().plot(
    kind="bar", color=["#55A868", "#C44E52", "#8C8C8C"]
)
plt.title("Overall Review Sentiment (VADER)")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

print(reviews_df["sentiment_label"].value_counts())
print()
print(reviews_df["compound_sentiment"].describe())

### 6.1.1 Sanity check: does sentiment track star rating?

In [ ]:
# If VADER is working sensibly on this domain (car-repair reviews, not
# movie/product reviews), average sentiment should rise steadily with stars.
sentiment_by_stars = reviews_df.groupby("stars")["compound_sentiment"].mean()

sentiment_by_stars.plot(kind="bar", color="#4C72B0")
plt.title("Average VADER Sentiment by Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Average Compound Sentiment")
plt.show()

print(sentiment_by_stars)
print()
print("Correlation (stars vs. compound sentiment):",
      round(reviews_df["stars"].corr(reviews_df["compound_sentiment"]), 3))

### 6.1.2 Sentiment by gender, within each star rating

Section 6.2 compares sentiment by gender overall, but stars and gender could both be driving that gap at once -- e.g. if one gender simply left more 5-star reviews, their average sentiment would look higher even if they write in an identical *tone* to the other gender at any given star level. Grouping by both **stars and gender together** controls for that: it asks whether men and women write with different sentiment even when they gave the exact same rating.

In [ ]:
# Mean sentiment, split by star rating AND gender at the same time.
by_star_gender = reviews_df.groupby(["stars", "inferred_gender"])["compound_sentiment"].agg(["mean", "count"])
print(by_star_gender)

In [ ]:
# Same numbers as a grouped bar chart -- easier to see where the two
# genders diverge (or don't) at each star level.
mean_table = reviews_df.groupby(["stars", "inferred_gender"])["compound_sentiment"].mean().unstack()
mean_table.plot(kind="bar", color=["#DD8452", "#4C72B0"])
plt.title("Average Sentiment by Star Rating, Split by Inferred Gender")
plt.xlabel("Star Rating")
plt.ylabel("Average Compound Sentiment")
plt.legend(title="Inferred Gender")
plt.show()

In [ ]:
# For each star level separately, test whether male vs. female sentiment
# differs at that level specifically (not just overall).
from scipy import stats

print(f"{'Stars':<7}{'n (male)':<10}{'n (female)':<12}{'mean (M)':<11}{'mean (F)':<11}{'t-stat':<9}{'p-value'}")
for star in sorted(reviews_df["stars"].unique()):
    subset = reviews_df[reviews_df["stars"] == star]
    m = subset.loc[subset["inferred_gender"] == "male", "compound_sentiment"]
    f = subset.loc[subset["inferred_gender"] == "female", "compound_sentiment"]
    t_stat, p_val = stats.ttest_ind(m, f, equal_var=False)
    print(f"{star:<7}{len(m):<10}{len(f):<12}{m.mean():<11.3f}{f.mean():<11.3f}{t_stat:<9.2f}{p_val:.4f}")

**Finding: a crossover pattern the overall comparison in Section 6.2 completely hides.**

At four of five star levels, female sentiment is higher than male sentiment, matching Section 6.2's aggregate result — but **at 1-star reviews, the pattern flips**: female sentiment (-0.293) is significantly *more negative* than male sentiment (-0.234) at that same rating (t = 3.29, p = 0.0010). Women's worst reviews run more negative than men's worst reviews, even though women's reviews are more positive on average overall.

| Stars | Significant? | Direction |
|---|---|---|
| 1 | Yes (p = 0.0010) | **Female more negative** |
| 2 | No (p = 0.844) | -- |
| 3 | No (p = 0.672) | -- |
| 4 | Yes (p = 0.0128) | Female more positive |
| 5 | Yes (p < 0.0001) | Female more positive |

**Why this matters more than Section 6.2's number:** the 1-star and 5-star groups are also the two largest (5,901 and 8,552 reviews respectively), so the overall gender-sentiment gap reported in Section 6.2 is being pulled upward by the 5-star group, where women are more positive by a wide margin — while masking that at the bottom (1-star, arguably the ratings most likely to reflect a genuinely bad experience, including a bad upsell interaction), women express *more* negativity than men, not less.

This reframes the takeaway from Section 6.2: it is not accurate to say "women's experiences are better on average." A more precise reading is that women's reviews are more emotionally polarized than men's — more positive at the top, more negative at the bottom — and the one place that polarization tips toward a worse outcome for women is exactly the rating tier where a bad upselling experience would most likely land.

## 6.2 Sentiment by Inferred Reviewer Gender

This is the core comparison the project is built around: does review sentiment differ between inferred-male and inferred-female reviewers of the same category of business?

In [ ]:
# compound_sentiment was computed by VADER back in Section 6.1 --
# this cell reuses that column, it does not recompute sentiment.
sentiment_by_gender = reviews_df.groupby("inferred_gender")["compound_sentiment"].describe()
print(sentiment_by_gender)

reviews_df.boxplot(column="compound_sentiment", by="inferred_gender")
plt.title("VADER Sentiment by Inferred Reviewer Gender")
plt.suptitle("")
plt.xlabel("Inferred Gender")
plt.ylabel("Compound Sentiment")
plt.show()

In [ ]:
# A t-test tells us whether the average sentiment gap between the two
# groups is bigger than what random chance alone would produce.
from scipy import stats

male_sentiment = reviews_df.loc[reviews_df["inferred_gender"] == "male", "compound_sentiment"]
female_sentiment = reviews_df.loc[reviews_df["inferred_gender"] == "female", "compound_sentiment"]

t_stat, p_value = stats.ttest_ind(male_sentiment, female_sentiment, equal_var=False)

print(f"Mean sentiment (male):   {male_sentiment.mean():.4f}  (n={len(male_sentiment):,})")
print(f"Mean sentiment (female): {female_sentiment.mean():.4f}  (n={len(female_sentiment):,})")
print(f"t-statistic: {t_stat:.3f}, p-value: {p_value:.4f}")

**Finding:** female reviewers show *higher* average sentiment (0.419) than male reviewers (0.359) — a statistically significant difference (t = -5.48, p < 0.0001), but in the **opposite direction** from a naive "women get worse treatment" hypothesis.

However, Section 6.1.2 already showed this aggregate number is misleading on its own: it's driven by the 5-star group, where women are considerably more positive, and it hides a reversal at 1-star, where women's sentiment is *more negative* than men's. This aggregate comparison should be read as a starting point, not the answer — the star-level breakdown and the upsell-keyword analysis below are what actually narrow in on the business question.

## 6.3 Upsell / Sales-Pressure Language Keyword Analysis

Sentiment alone doesn't tell us *why* a review is negative or positive — it can't distinguish "the mechanic was rude" from "they tried to sell me services I didn't need." To get at the actual business question, reviews are flagged if they contain language associated with upselling or sales pressure. Before building that list, the cell below reads a real sample of reviews directly, to check what this language actually looks like rather than assuming it.

### 6.3.1 Manual read: what does upsell language actually look like in this data?

Before building a keyword list, it's worth reading a sample of real reviews directly, rather than assuming what upsell language looks like. Earlier in this project, a similar manual read of a small sample (on an earlier, pre-PA version of the data) surfaced phrases like "not pushy" and "assuming they would try to sell me one either way" — that read is what originally motivated this section. It wasn't reproducible from inside the notebook, though. The cell below re-does it on this exact PA dataset, with a fixed random seed, so anyone re-running this notebook can read the same sample and check the claim themselves instead of taking it on faith.

In [ ]:
# A reproducible random sample of real reviews, read directly -- this is
# what actually motivates the keyword list in the next code cell, not just an
# assumption about what upsell language sounds like.
sample_for_reading = reviews_df.sample(n=15, random_state=7)[["stars", "inferred_gender", "text"]]

for _, row in sample_for_reading.iterrows():
    print(f"[{row['stars']} stars, inferred {row['inferred_gender']}]")
    print(row["text"])
    print("-" * 80)

**Reading through this real sample (seed=7) surfaced two honest findings, not just a confirmation of the keyword approach:**

**1. The keyword list misses real upsell/pressure complaints that don't use its exact phrasing.** Only one review in this sample of 15 actually triggers a current keyword ("...completely taken advantage of and pressured into buying a car..."). But at least three others clearly describe upsell-adjacent situations without matching any keyword: an undisclosed $20 alignment-check charge ("I should have been made aware of the charge"), a shop that "fix[es] something and charge[s] you $1,000 without asking for approval," and a bait-and-switch "free oil change" review that explicitly says "Can you say BAIT AND SWITCH?" — none of which contain words like "unnecessary," "pressure," or "upsell." This is direct, concrete evidence of a real recall gap: any fixed keyword list built from a read like this one will undercount how often this actually happens.

**2. A visible gender-inference error.** The review describing being "pressured into buying a car" while "being pregnant with twins" — someone who refers to herself as a "pregnant women [sic]" — was inferred as **male** by this project's gender-inference pipeline. This is a concrete, unambiguous example of the method's real error rate, not just an abstract caveat. It doesn't invalidate the overall approach (Section 5.1 already showed gender-guesser + NLTK NB isn't perfect), but it's worth citing directly here as the kind of individual-level mistake that an aggregate accuracy number can hide.

**Takeaway for Section 7:** both findings point the same direction — a fixed keyword list is a reasonable first pass but visibly undercounts real cases, and gender inference from first names alone will misclassify some real people. A trained classifier for upsell language (rather than exact-phrase matching) and a plan for validating/bounding the gender-inference error rate are both worth prioritizing in the next stage.

In [ ]:
# Keyword list. Terms like "pushy", "sell me", "pressure(d)", and
# "convince" trace to phrases actually seen in an early manual read of
# real reviews (before the PA-only restriction was applied). The rest
# ("unnecessary", "quoted me", "upsell", "talk me into", etc.) were added
# from general reasoning about what this language sounds like, then
# spot-checked against real matches in this dataset below (Section 6.3.2)
# before being trusted.
import re

upsell_keywords = [
    r"upsell", r"up-sell", r"up sell",
    r"push(?:y|ed|ing)?", r"pressure", r"pressured",
    r"tried to sell", r"trying to sell", r"sell me",
    r"unnecessary", r"didn'?t need", r"did not need",
    r"recommended (?:a|an|additional|more)", r"suggested (?:a|an|additional|more)",
    r"quoted me", r"extra (?:charge|cost|fee|service)",
    r"talk(?:ed)? me into", r"convince(?:d)? me",
]
upsell_re = re.compile("|".join(upsell_keywords), flags=re.IGNORECASE)


def mentions_upsell(text):
    return bool(upsell_re.search(str(text)))


reviews_df["upsell_mention"] = reviews_df["text"].apply(mentions_upsell)

print(f"Reviews mentioning upsell/pressure language: "
      f"{reviews_df['upsell_mention'].sum():,} / {len(reviews_df):,} "
      f"({reviews_df['upsell_mention'].mean():.1%})")

### 6.3.2 Spot-checking the keyword list against real matches

Before trusting the counts above, each keyword is checked against actual matching text from this dataset — not just assumed to work.

In [ ]:
# Per-keyword hit counts, plus real example text for the terms that were
# NOT confirmed by the earlier manual read (added from general reasoning).
import collections

hit_counts = collections.Counter()
for text in reviews_df["text"].astype(str):
    for kw in upsell_keywords:
        if re.search(kw, text, flags=re.IGNORECASE):
            hit_counts[kw] += 1

print("Hits per keyword pattern:")
for kw, n in hit_counts.most_common():
    print(f"{n:5d}  {kw}")

print()
print("Example matches for terms added from general reasoning (not the original manual read):")
for label, pattern in [
    ("unnecessary", r"unnecessary"),
    ("quoted me", r"quoted me"),
    ("upsell", r"upsell|up-sell|up sell"),
    ("talk me into", r"talk(?:ed)? me into"),
]:
    example = reviews_df.loc[
        reviews_df["text"].astype(str).str.contains(pattern, case=False, regex=True, na=False), "text"
    ].iloc[0]
    print(f"\n[{label}]  ...{str(example)[:180]}...")

In [ ]:
# Does the rate of upsell-language mentions differ by inferred gender?
# A chi-square test checks whether that difference is bigger than chance.
mention_table = pd.crosstab(reviews_df["inferred_gender"], reviews_df["upsell_mention"])
print(mention_table)
print()

mention_rate_by_gender = reviews_df.groupby("inferred_gender")["upsell_mention"].mean()
mention_rate_by_gender.plot(kind="bar", color=["#4C72B0", "#DD8452"])
plt.title("Upsell/Pressure Mention Rate by Inferred Gender")
plt.xlabel("Inferred Gender")
plt.ylabel("Share of Reviews Mentioning Upsell Language")
plt.show()

print(mention_rate_by_gender)
print()

chi2, chi2_p, dof, expected = stats.chi2_contingency(mention_table)
print(f"Chi-square: {chi2:.3f}, p-value: {chi2_p:.4f}")

In [ ]:
# Among ONLY the reviews that mention upsell/pressure language, is the
# sentiment itself more negative for one gender than the other?
flagged_df = reviews_df[reviews_df["upsell_mention"]]

flagged_sentiment_by_gender = flagged_df.groupby("inferred_gender")["compound_sentiment"].describe()
print(flagged_sentiment_by_gender)

male_flagged = flagged_df.loc[flagged_df["inferred_gender"] == "male", "compound_sentiment"]
female_flagged = flagged_df.loc[flagged_df["inferred_gender"] == "female", "compound_sentiment"]

t_stat_flagged, p_value_flagged = stats.ttest_ind(male_flagged, female_flagged, equal_var=False)
print(f"\nWithin upsell-flagged reviews only:")
print(f"Mean sentiment (male):   {male_flagged.mean():.4f}  (n={len(male_flagged):,})")
print(f"Mean sentiment (female): {female_flagged.mean():.4f}  (n={len(female_flagged):,})")
print(f"t-statistic: {t_stat_flagged:.3f}, p-value: {p_value_flagged:.4f}")

**Finding (sentiment within flagged reviews):** among reviews that already mention upsell language, sentiment is *not* significantly different by gender (t = -1.55, p = 0.12). Female-flagged reviews average slightly higher sentiment (0.451) than male-flagged reviews (0.401), but at this sample size (917 vs. 1,101) that gap isn't distinguishable from chance. So while women are somewhat more likely to bring up upsell language at all, once they do, the resulting review isn't measurably more negative than a man's upsell-flagged review.

**Putting Section 6.2 and 6.3 together:** of the three comparisons tested — overall sentiment by gender, upsell-mention rate by gender, and sentiment within upsell-flagged reviews by gender — only the mention-rate gap is both statistically significant and consistent with the original hypothesis. That is a narrower, more specific finding than "women have worse experiences at auto repair shops": it suggests any real effect is about *how often* upselling comes up in the conversation, not about how customers feel once it does.

**Limitation to flag explicitly:** this keyword list combines phrases actually observed in the manual read above (`pushy`, `sell me`, `pressure(d)`, `convince`; Section 6.3.1) with terms added from general reasoning and confirmed only by a spot-check of real matches (Section 6.3.2), not a systematic labeled dataset. It will still miss paraphrased upsell complaints (e.g. a review that describes the situation without using any of these words) and may over-count incidental, non-upsell uses of common words like "recommended." Section 7 (Proposed AI Solution) addresses building and validating a proper classifier for this instead of a fixed keyword list.

## 6.4 Preliminary Text Classification

As a preliminary check that this review text is usable for supervised text classification (the technique planned for Section 7), a simple bag-of-words + Naive Bayes model — the same approach as the course's `Sentiment_Analysis.ipynb` — is tested here predicting whether a review is positive (4-5 stars) or negative (1-2 stars) from its text alone. 3-star reviews are excluded as genuinely mixed/neutral.

In [ ]:
# Reuse the exact CountVectorizer + Naive Bayes recipe from
# Sentiment_Analysis.ipynb, just applied to our own review text.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from nltk.tokenize import RegexpTokenizer

polar_df = reviews_df[reviews_df["stars"] != 3].copy()
polar_df["label"] = (polar_df["stars"] >= 4).astype(int)  # 1 = positive, 0 = negative

X_train, X_test, y_train, y_test = train_test_split(
    polar_df["text"], polar_df["label"], test_size=0.2, random_state=5
)

token = RegexpTokenizer(r"[a-zA-Z]+")
cv = CountVectorizer(stop_words="english", ngram_range=(1, 1), tokenizer=token.tokenize, max_features=800)
X_train_vect = cv.fit_transform(X_train)
X_test_vect = cv.transform(X_test)

nb_text_classifier = MultinomialNB()
nb_text_classifier.fit(X_train_vect, y_train)
predicted = nb_text_classifier.predict(X_test_vect)

print(metrics.classification_report(y_test, predicted, target_names=["negative", "positive"]))

**Finding:** a simple 800-feature bag-of-words Naive Bayes classifier reaches **93% accuracy** separating positive (4-5 star) from negative (1-2 star) reviews using text alone. This suggests auto-repair review language is polarized and predictable (plausibly because service failures get described in more formulaic terms — "overcharged," "unnecessary," "honest").

This matters for the next stage: it confirms the review text itself carries a strong, learnable signal, which supports building a proper trained classifier for upsell-language detection in Section 7, rather than relying on the fixed keyword list used in Section 6.3.

## 6.5 LLM-Assisted Upsell Labeling (Voyager LLM API)

Section 6.3.1 showed the keyword list misses real upsell complaints (for example, a shop that "charge[s] you $1,000 without asking for approval"). This section measures how big that gap is. An LLM, served through ASU Research Computing's Voyager LLM API (OpenAI-compatible), labels a sample of reviews, and its labels are compared with the keyword flag.

**Design:** 100 keyword-flagged and 100 unflagged reviews (fixed seed). Sampling both groups lets us estimate how often the keywords are right (LLM agreement among flagged reviews) and how often they miss (LLM positives among unflagged reviews). Population-level estimates re-weight the two groups by their true sizes.

**What this is not:** the LLM's labels are a proxy, not ground truth. Agreement with the keyword list does not prove either one correct, so Section 6.5.4 prints the disagreements to be read by hand.

**Running it:** paste your Voyager API key into the `API_KEY` placeholder in Section 6.5.1 (or leave the placeholder and set the `OPENAI_API_KEY` environment variable instead). Clear the key again before saving or submitting, because a key left in the notebook or its HTML export is shared with it. Labels are cached in `llm_upsell_labels.csv`, so the notebook still runs, and shows the saved results, for anyone without a key.

### 6.5.1 Sample, definition, and prompt

In [ ]:
# Settings for the ASU Research Computing (Voyager) LLM API.
import os
import re
import json
import time
import requests

# ============================ WHERE TO PASTE YOUR KEY ============================
# Replace the words inside the quotes on the API_KEY line below with your Voyager
# key. That is the ONLY line to edit -- leave everything else alone.
# Put the words back (delete the key) before you save, export, or submit: a key
# left in this notebook or its HTML export is shared along with it.
# ================================================================================
API_KEY = ""  # supply privately through environment configuration
if not API_KEY.strip() or " " in API_KEY.strip():
    # Still a placeholder phrase (real keys contain no spaces): try the environment variable.
    API_KEY = os.environ.get("OPENAI_API_KEY", "")
HAVE_KEY = bool(API_KEY.strip())
print(f"API key detected: {HAVE_KEY}  (the key itself is never printed)")

LLM_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://openai.rc.asu.edu/v1")
LLM_MODEL = os.environ.get("LLM_MODEL", "gemma4-31b-it")  # confirm the name via GET /v1/models
LLM_CACHE = "llm_upsell_labels.csv"
N_PER_GROUP = 100

# Stratified sample: 100 reviews the keyword list flagged + 100 it did not.
# Sampling both groups lets us measure what the keywords MISS, not just what they hit.
sample_flagged = reviews_df[reviews_df["upsell_mention"]].sample(n=N_PER_GROUP, random_state=11)
sample_unflagged = reviews_df[~reviews_df["upsell_mention"]].sample(n=N_PER_GROUP, random_state=11)
llm_sample = pd.concat([sample_flagged, sample_unflagged])[
    ["review_id", "stars", "inferred_gender", "upsell_mention", "text"]
].reset_index(drop=True)

SYSTEM_PROMPT = "You label customer reviews of auto repair businesses. Reply with JSON only."


def build_user_prompt(review_text):
    # The definition below is exactly what the LLM is judged against.
    return (
        "An 'upsell or sales-pressure complaint' means the reviewer says staff recommended, "
        "pushed, or charged for services or products beyond what the reviewer asked for or "
        "needed, pressured them into a purchase, charged without approval, or used a "
        "bait-and-switch on price. A suggestion the reviewer welcomed (for example "
        "'they suggested an oil change, great!') does NOT count. Rude service, long waits, "
        "or high prices alone do NOT count.\n\n"
        "Review:\n" + review_text + "\n\n"
        'Reply with only this JSON: {"upsell": true or false, "evidence": "short quote from the review, or empty"}'
    )


def parse_llm_label(raw):
    """Take the LAST {...} object in the reply (models sometimes think out loud first)."""
    objs = re.findall(r"\{[^{}]*\}", raw or "")
    if not objs:
        return None, ""
    try:
        data = json.loads(objs[-1])
        value = data["upsell"]
    except (ValueError, KeyError):
        return None, ""
    if isinstance(value, str):
        value = value.strip().lower() == "true"
    return bool(value), str(data.get("evidence", ""))[:200]


def label_review(review_text, max_retries=5):
    headers = {"Authorization": "Bearer " + API_KEY}
    body = {
        "model": LLM_MODEL,
        "temperature": 0,
        "max_tokens": 800,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(review_text)},
        ],
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(LLM_BASE_URL + "/chat/completions", headers=headers, json=body, timeout=120)
        except requests.exceptions.RequestException:
            time.sleep(2 ** attempt)  # network hiccup: wait, then retry
            continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(2 ** attempt)  # rate limited or server busy: wait, then retry
            continue
        if resp.status_code >= 400:
            raise RuntimeError(f"LLM API error {resp.status_code}: {resp.text[:300]}")
        return parse_llm_label(resp.json()["choices"][0]["message"]["content"])
    return None, ""

print(f"Sample: {len(llm_sample)} reviews ({N_PER_GROUP} keyword-flagged, {N_PER_GROUP} unflagged)")

### 6.5.2 Get the labels (saved file or live API)

In [ ]:
# Use saved labels if they match this exact sample; otherwise call the API
# (needs the API key from Section 6.5.1). Saving to a CSV means the notebook
# still runs for anyone without a key.
llm_df = None
if os.path.exists(LLM_CACHE):
    cached = pd.read_csv(LLM_CACHE)
    if set(cached["review_id"]) == set(llm_sample["review_id"]):
        llm_df = cached
        print(f"Loaded {len(llm_df)} saved LLM labels from {LLM_CACHE}")

if llm_df is None and HAVE_KEY:
    results = []
    for i, review_text in enumerate(llm_sample["text"]):
        results.append(label_review(str(review_text)[:3000]))
        if (i + 1) % 25 == 0:
            print(f"  labeled {i + 1} / {len(llm_sample)}")
        time.sleep(0.5)  # gentle pacing: the docs don't publish rate limits
    llm_df = llm_sample.copy()
    llm_df["llm_upsell"] = [r[0] for r in results]
    llm_df["llm_evidence"] = [r[1] for r in results]
    n_failed = int(llm_df["llm_upsell"].isna().sum())
    if n_failed == 0:
        llm_df.to_csv(LLM_CACHE, index=False)
        print(f"Saved labels to {LLM_CACHE}")
    else:
        print(f"{n_failed} reviews could not be labeled; cache not saved so a re-run retries them.")

if llm_df is None:
    print("No saved labels and no API key set, so the LLM step is skipped.")
    print("To run it: create a key in the Voyager portal (LLM Access tab) and paste it into")
    print("API_KEY in Section 6.5.1 (or set the OPENAI_API_KEY environment variable).")

### 6.5.3 Agreement with the keyword list

In [ ]:
# How often does the LLM agree with the keyword flag, and how many real
# cases do the keywords miss? Estimates are relative to the LLM's labels.
if llm_df is not None:
    scored = llm_df.dropna(subset=["llm_upsell"]).copy()
    scored["llm_upsell"] = scored["llm_upsell"].astype(bool)

    print("Keyword flag (rows) vs. LLM label (columns):")
    print(pd.crosstab(scored["upsell_mention"], scored["llm_upsell"]))
    print()

    flagged = scored[scored["upsell_mention"]]
    unflagged = scored[~scored["upsell_mention"]]
    p_flagged = flagged["llm_upsell"].mean()      # keyword hits the LLM agrees with
    p_unflagged = unflagged["llm_upsell"].mean()  # keyword misses the LLM catches

    # Re-weight the two groups by their real sizes in the full 16,735 reviews.
    n_flagged_all = int(reviews_df["upsell_mention"].sum())
    n_unflagged_all = len(reviews_df) - n_flagged_all
    est_llm_positives = n_flagged_all * p_flagged + n_unflagged_all * p_unflagged

    print(f"Keyword-flagged reviews the LLM also calls upsell: "
          f"{int(flagged['llm_upsell'].sum())} / {len(flagged)} ({p_flagged:.1%})")
    print(f"Unflagged reviews the LLM calls upsell (keyword misses): "
          f"{int(unflagged['llm_upsell'].sum())} / {len(unflagged)} ({p_unflagged:.1%})")
    if est_llm_positives > 0:
        print(f"Estimated share of ALL reviews the LLM would call upsell: "
              f"{est_llm_positives / len(reviews_df):.1%}  (keyword list flags {n_flagged_all / len(reviews_df):.1%})")
        print(f"Estimated keyword recall against the LLM's labels: "
              f"{n_flagged_all * p_flagged / est_llm_positives:.1%}")
else:
    print("(skipped: no LLM labels available)")

### 6.5.4 Disagreements to read by hand

In [ ]:
# The LLM is a proxy, not ground truth -- read the cases where it and the
# keyword list disagree and judge for yourself which one is right.
if llm_df is not None:
    misses = scored[(~scored["upsell_mention"]) & scored["llm_upsell"]].head(5)
    false_alarms = scored[scored["upsell_mention"] & (~scored["llm_upsell"])].head(5)

    for title, group in [("KEYWORDS MISSED, LLM SAYS UPSELL", misses),
                         ("KEYWORDS FLAGGED, LLM SAYS NOT UPSELL", false_alarms)]:
        print("=" * 80)
        print(title)
        for _, row in group.iterrows():
            print(f"\n[{row['stars']} stars] LLM evidence: {row['llm_evidence']}")
            print(str(row["text"])[:500].replace("\n", " "))
else:
    print("(skipped: no LLM labels available)")

**Interpretation:** *(To be written after the first run with a key. It should report the keyword list's estimated precision and recall against the LLM labels, the LLM's estimated overall upsell rate, and what the hand-read disagreements show, including any case where the LLM rather than the keywords was wrong.)*

# 7. Proposed AI Solution

Section 6 pointed at three specific gaps that a fixed keyword list and a first-name lookup can't close on their own. The next stage addresses each directly:

**1. A trained classifier for upsell/pressure language, replacing the keyword list.** Section 6.3.1's manual read showed real upsell complaints ("charge you $1,000 without asking for approval," "BAIT AND SWITCH") that the keyword list in Section 6.3 completely misses because they don't use its exact phrasing. The next stage hand-labels a sample of reviews as upsell-mentioned / not, then trains a supervised classifier (starting with the TF-IDF + Naive Bayes / SVM approach already validated on this data in Section 6.4, and comparing it against a transformer-based text embedding model such as `sentence-transformers`) to recognize the *concept* of an unwanted upsell rather than specific words. This should substantially close the recall gap Section 6.3.1 demonstrated, and it can be evaluated properly with precision/recall against the labeled sample — something a keyword list can't be graded on at all.

**2. A bounded estimate of the gender-inference error rate.** Section 6.3.1 also surfaced a concrete misclassification (a reviewer who identifies herself as pregnant, inferred as male). Rather than treating that as an isolated anecdote, the next stage hand-labels a random sample of reviewer names against their review text (where gender is inferable from context, e.g. "my husband," "as a woman") to estimate a real precision/recall for the `gender-guesser` + NLTK hybrid built in Section 5.1, instead of relying only on the small 25-name comparison already run there.

**3. Re-running the Section 6 comparisons with the improved classifier and a known error bound.** Once (1) and (2) exist, the same statistical comparisons already built in Section 6 — mention rate by gender (chi-square), sentiment within flagged reviews (t-test), and the star-controlled crossover check from Section 6.1.2 — get re-run on the higher-recall classifier output, with the gender-inference error rate reported alongside the results as a confidence bound rather than an unstated assumption.

**What this is expected to produce:** a validated, reproducible pipeline that goes from raw Yelp review text to a defensible answer on whether upsell language differs by inferred reviewer gender in this metro area — plus the honest error bars needed to say how much to trust that answer. That combination (finding + its own uncertainty) is what would actually be usable by the business audiences named in Section 2: a franchise quality-assurance team could act on a flagged pattern only if they also know how often the flag itself is wrong.

# 8. Generative AI Use

Claude (Anthropic's Claude Code) was used throughout this project as a coding and analysis aid, including:

- Assessing project feasibility and identifying what data (business/review/user files, and later the PA-only restriction) would be needed before writing any code.
- Writing and debugging the data pipeline in Section 3 (filtering, streaming large files, joining datasets).
- Suggesting and implementing the gender-inference comparison methodology in Section 5.1, including running the empirical comparison between `gender-guesser` and an NLTK Naive Bayes classifier that led to the hybrid approach used.
- Writing and executing the sentiment analysis, statistical tests (t-tests, chi-square), and keyword-analysis code in Section 6, including the star-controlled comparison in Section 6.1.2 and the manual-read validation in Section 6.3.1.
- Drafting written explanations and interpretations of results throughout the notebook.

All code, statistical results, and conclusions in this notebook were reviewed and are understood by the team. Where Claude's suggested comment or claim was found to overstate what the data actually showed (for example, an early draft comment overstating how the upsell keyword list was built), it was corrected and re-validated against the real dataset rather than accepted as written — Section 6.3.1's spot-check exists specifically because of that correction.

In [ ]:
!jupyter nbconvert --to html ProjectEDA_Automotive_Repair.ipynb
